# 02. 피처 실험

- 담당: **② 홍시윤 (데이터·피처)** (이 파일은 담당자만 수정)
- 기록 파일: `experiments/log_H_data.csv`

### 할 일
- [ ] 기본 피처(WT=0, 변이=1)와 비교할 새 피처 만들기 (변이 개수, 변이 종류 구분 등)
- [ ] `run_cv()`로 기본 피처보다 오르는지 확인 → 오르면 `src/features.py`로 옮기기
- [ ] 피처 선택은 반드시 Pipeline 안에서 (fold 안에서만 fit)
- [ ] **화요일 12시 피처 확정**

> **규칙:** 전처리는 Pipeline 안에 / test는 보지 않기 / 점수는 `run_cv()`로 / 결과는 `log_result()`로 기록 / 자세한 규칙은 `docs/coding_rules.md`

## 0. 환경 설정 (수정하지 말고 그대로 실행)
Colab은 구글 드라이브 `내 드라이브`에 레포 폴더(`ah07-cancer-subtype`, 안에 `data/` 포함)를 두면 돼요.

In [37]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/ah07-cancer-subtype"  # 드라이브에 둔 레포 폴더
else:
    REPO_DIR = os.path.abspath(os.path.join("..", ".."))  # notebooks/이니셜_역할/ 폴더 기준

# 경로 확인: 여기서 멈추면 폴더 위치나 이름을 확인해 주세요
assert os.path.isdir(os.path.join(REPO_DIR, "src")), (
    f"src 폴더를 찾을 수 없어요: {REPO_DIR}\n"
    "→ 드라이브 '내 드라이브' 바로 아래에 'ah07-cancer-subtype' 폴더가 있는지, "
    "ZIP을 풀 때 폴더가 두 번 겹치지 않았는지(ah07-cancer-subtype/ah07-cancer-subtype-dev) 확인")
assert os.path.isfile(os.path.join(REPO_DIR, "data", "train.csv")), (
    "data/train.csv 가 없어요 → 데이콘 csv 3개를 data/ 폴더에 넣어 주세요")
sys.path.insert(0, REPO_DIR)

from src.config import SEED, DATA_DIR, set_seed
set_seed(SEED)
print("REPO_DIR:", REPO_DIR, "| DATA_DIR:", DATA_DIR)

REPO_DIR: /Users/admin/Documents/ah07-cancer-subtype | DATA_DIR: /Users/admin/Documents/ah07-cancer-subtype/data


In [38]:
import platform
import numpy as np, pandas as pd, sklearn
from sklearn.pipeline import Pipeline

from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import basic_transformer, binarize_mutations
from src.features import mutation_count
from src.validation import get_folds, run_cv, balanced_weights
from src.evaluation import macro_f1, per_class_f1, log_result
from src.prediction import make_submission

print("OS:", platform.platform(), "| python", platform.python_version(),
      "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

OS: macOS-15.7.3-arm64-arm-64bit-Mach-O | python 3.14.8 | pandas 3.1.1 | scikit-learn 1.9.1


## 1. 데이터

In [39]:
train, test = load_data(DATA_DIR)   # test는 형식 확인과 최종 추론에만
X, y_raw = split_xy(train)
y, le = encode_target(y_raw)
folds = get_folds(y)                 # 팀 공통 fold
print("train", train.shape, "| test", test.shape, "| classes", len(le.classes_))

train (6201, 4386) | test (2546, 4385) | classes 26


## 실험 예시 (복사해서 바꿔 쓰세요)
1. Pipeline에 피처 변환과 모델을 넣고
2. `run_cv()`로 점수를 내고
3. `log_result()`로 기록해요.

In [40]:
# from xgboost import XGBClassifier
# PARAMS = dict(n_estimators=200, learning_rate=0.1, max_depth=6, tree_method="hist", random_state=SEED, n_jobs=-1)
# pipe = Pipeline([("features", basic_transformer()), ("model", XGBClassifier(**PARAMS))])
# cv = run_cv(pipe, X, y, folds=folds)
# per_class_f1(y, cv["oof"], le.classes_).head(10)
# log_result(owner="H_data", notebook="02_features.ipynb", experiment="실험 이름",
#            features="basic(WT=0,변이=1)", model="XGBoost", params=PARAMS, cv=cv, notes="")

=============================공통 코드 끝=============================

## 2. 피처 구성 방법별 실험

### A. 원본 인코딩

In [41]:
# 원본 데이터에서 변이 문자열 추출
mutations = X.stack()

# 결측치와 WT 제거
mutations = mutations.dropna()
mutations = mutations[mutations.ne("WT")]

# 공백으로 구분된 복수 변이 확인
multi_examples = mutations[
    mutations.astype("string").str.contains(r"\s+", regex=True, na=False)
]

print("복수 변이가 기록된 셀 수:", len(multi_examples))
print("\n복수 변이 예시:")
print(multi_examples.head(10).to_string())

복수 변이가 기록된 셀 수: 22026

복수 변이 예시:
2  ITGB5        Q369* I368N
   LAMB3        S622S G827R
   MYH4        E412K R1800C
   RYR1       D3546V G3739G
   TNC          A368V E646K
5  ADAMTS1      C521Y R258R
   ADCY9        G273R V155A
   ALDH3A1       F336L I71T
   ANO1        I758S P700fs
   BCAN         R107H E468E


In [42]:
# A. 유전자-변이 조합 추출 함수
def extract_mutation_pairs(X):
    pairs = []

    # 샘플별로 유전자 컬럼 확인
    for _, row in X.iterrows():
        sample_pairs = []

        for gene, value in row.items():
            # 결측치 또는 WT는 제외
            if pd.isna(value) or value == "WT":
                continue

            # 공백 기준으로 개별 변이 분리
            for mutation in str(value).split():
                sample_pairs.append(f"{gene}_{mutation}")

        pairs.append(sample_pairs)

    return pairs


# 원본 train 데이터에서 유전자-변이 조합 추출
mutation_pairs = extract_mutation_pairs(X)

print("첫 번째 샘플의 변이 조합:")
print(mutation_pairs[0][:10])

print("\n전체 샘플 수:", len(mutation_pairs))

첫 번째 샘플의 변이 조합:
['ABCC2_D623D', 'ADCY2_F736C', 'CD86_P117R', 'EPHB4_R305S', 'FOXC2_N23K', 'GBF1_L192V', 'GLMN_G592G', 'GNAI1_K277K', 'GOLGA4_A699T', 'GPR182_L111L']

전체 샘플 수: 6201


ex) 

ABCC2_D623D → ABCC2 유전자의 D623D 변이

ADCY2_F736C → ADCY2 유전자의 F736C 변이

전체 샘플 수 6,201개 → 원본 train 데이터와 일치

In [43]:
from collections import Counter

import numpy as np
from scipy.sparse import csr_matrix
from sklearn.base import BaseEstimator, TransformerMixin


class MutationMultiHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, min_frequency=5):
        self.min_frequency = min_frequency

    def fit(self, X, y=None):
        # 학습 fold에서만 변이 조합별 등장 빈도 계산
        mutation_pairs = extract_mutation_pairs(X)

        # 같은 샘플에 동일한 변이가 중복되어도 1회만 집계
        counts = Counter(
            pair
            for sample in mutation_pairs
            for pair in set(sample)
        )

        # 학습 fold에서 5개 이상 샘플에 등장한 변이만 유지
        selected = sorted(
            pair for pair, count in counts.items()
            if count >= self.min_frequency
        )

        # 선택된 변이 조합과 피처 인덱스 저장
        self.feature_names_ = selected
        self.feature_to_idx_ = {
            pair: idx for idx, pair in enumerate(selected)
        }

        return self

    def transform(self, X):
        # fit에서 선택된 피처만 사용
        mutation_pairs = extract_mutation_pairs(X)

        rows = []
        cols = []

        for row_idx, sample in enumerate(mutation_pairs):
            for pair in set(sample):
                col_idx = self.feature_to_idx_.get(pair)

                if col_idx is not None:
                    rows.append(row_idx)
                    cols.append(col_idx)

        # 존재하는 변이 조합은 1, 나머지는 0
        data = np.ones(len(rows), dtype=np.float32)

        return csr_matrix(
            (data, (rows, cols)),
            shape=(len(mutation_pairs), len(self.feature_names_)),
            dtype=np.float32
        )

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.feature_names_, dtype=object)

In [44]:
import pandas as pd

# 변환 동작을 확인하기 위한 가상 데이터
X_sample = pd.DataFrame({
    "TP53": [
        "R175H R248Q",
        "R175H",
        "R175H",
        "WT",
        None
    ],
    "EGFR": [
        "L858R",
        "WT",
        "L858R",
        "L858R",
        "WT"
    ]
})

# 예제 데이터이므로 빈도 기준을 2로 설정
test_encoder = MutationMultiHotEncoder(min_frequency=2)

# 예제 데이터로만 학습 및 변환
X_sample_encoded = test_encoder.fit_transform(X_sample)

# 결과 확인
result = pd.DataFrame(
    X_sample_encoded.toarray(),
    columns=test_encoder.get_feature_names_out()
)

display(result)

print("선택된 피처:", test_encoder.get_feature_names_out().tolist())
print("변환 결과 크기:", X_sample_encoded.shape)

,EGFR_L858R,TP53_R175H
0,1.0,1.0
1,0.0,1.0
2,1.0,1.0
3,1.0,0.0
4,0.0,0.0


선택된 피처: ['EGFR_L858R', 'TP53_R175H']
변환 결과 크기: (5, 2)


In [45]:
from xgboost import XGBClassifier
from src.config import SEED

# B·C·D 실험과 동일한 XGBoost 설정
base_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    tree_method="hist",
    random_state=SEED,
    n_jobs=-1
)

print("base_model 생성 완료")

base_model 생성 완료


In [46]:
from sklearn.pipeline import Pipeline
from sklearn.base import clone

pipeline_A = Pipeline([
    ("features", MutationMultiHotEncoder(min_frequency=5)),
    ("model", clone(base_model))
])

print("A 모델 설정:", pipeline_A.named_steps["model"].get_params())
print("A 변환기:", pipeline_A.named_steps["features"])

A 모델 설정: {'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': None, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': True, 'eval_metric': None, 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.1, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 6, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 200, 'n_jobs': -1, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': None, 'tree_method': 'hist', 'validate_parameters': None, 'verbosity': None}
A 변환기: MutationMultiHotEncoder()


In [47]:
# 기존 공통 fold 사용
train_idx, valid_idx = next(iter(folds))

# 첫 번째 학습 fold만 사용해 인코더 학습
encoder_check = MutationMultiHotEncoder(min_frequency=5)
encoder_check.fit(X.iloc[train_idx])

# 학습 fold와 검증 fold 변환
X_train_A = encoder_check.transform(X.iloc[train_idx])
X_valid_A = encoder_check.transform(X.iloc[valid_idx])

print("선택된 피처 수:", len(encoder_check.feature_names_))
print("학습 데이터 크기:", X_train_A.shape)
print("검증 데이터 크기:", X_valid_A.shape)
print("희소행렬 형식:", type(X_train_A).__name__)
print("학습 데이터의 1 개수:", X_train_A.nnz)

선택된 피처 수: 284
학습 데이터 크기: (4960, 284)
검증 데이터 크기: (1241, 284)
희소행렬 형식: csr_matrix
학습 데이터의 1 개수: 3456


In [48]:
from src.data_loader import encode_target

# 암종 레이블을 숫자로 변환
y_encoded, label_encoder = encode_target(y)

print("인코딩된 타깃 크기:", y_encoded.shape)
print("암종 클래스 수:", len(label_encoder.classes_))

인코딩된 타깃 크기: (6201,)
암종 클래스 수: 26


In [49]:
from src.validation import get_folds, run_cv

# 기존 B·C·D와 동일한 교차검증 분할
folds = get_folds(y_encoded)

print("Fold 수:", len(folds))

Fold 수: 5


### B. 이진 인코딩

In [50]:
# B. 이진 인코딩 결과 확인 (WT=0, 변이=1)

X_B = binarize_mutations(X)

print("원본 데이터 크기:", X.shape)
print("이진 인코딩 데이터 크기:", X_B.shape)
print("변환 후 고유값:", np.unique(X_B.to_numpy()))

display(X_B.iloc[:5, :10])

원본 데이터 크기: (6201, 4384)
이진 인코딩 데이터 크기: (6201, 4384)
변환 후 고유값: [0 1]


,A2M,AAAS,AADAT,AARS1,ABAT,ABCA1,ABCA2,ABCA3,ABCA4,ABCA5
0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0


### C. 이진 인코딩 + 변이 개수

In [51]:
# C. 이진 인코딩 + 환자별 변이 유전자 개수 피처 생성

# B 방식의 이진 인코딩 결과 복사
X_C = X_B.copy()

# 환자별 변이 유전자 개수 추가
X_C["n_mutated_genes"] = mutation_count(X)

# 결과 확인
print("B 방식 데이터 크기:", X_B.shape)
print("C 방식 데이터 크기:", X_C.shape)

print("환자별로 변이가 기록된 유전자 수")
display(X_C[["n_mutated_genes"]].head(10))

B 방식 데이터 크기: (6201, 4384)
C 방식 데이터 크기: (6201, 4385)
환자별로 변이가 기록된 유전자 수


,n_mutated_genes
0,25
1,17
2,119
3,7
4,34
5,270
6,4
7,1
8,9
9,20


In [52]:
# C 방식의 변이 유전자 개수 피처 검증

count_check = X_B.sum(axis=1)

print(
    "변이 개수 일치 여부:",
    (count_check == X_C["n_mutated_genes"]).all()
)

print(
    "결측치 개수:",
    X_C["n_mutated_genes"].isna().sum()
)

변이 개수 일치 여부: True
결측치 개수: 0


### D. 이진 인코딩 + 변이 개수 + 변이 유형별 개수

#### D

In [53]:
# D. 변이 유형별 개수 피처 생성을 위한 원본 변이값 확인

# WT와 결측치를 제외한 실제 변이 기록만 추출
mutation_values = X.stack()
mutation_values = mutation_values[mutation_values != "WT"]

print("전체 변이 기록 셀 수:", len(mutation_values))
print("서로 다른 변이 표기 수:", mutation_values.nunique())

print("\n가장 자주 등장하는 변이 표기 TOP 30")
display(mutation_values.value_counts().head(30))

print("\n변이 표기 예시 30개")
print(mutation_values.drop_duplicates().head(30).tolist())

전체 변이 기록 셀 수: 218893
서로 다른 변이 표기 수: 118846

가장 자주 등장하는 변이 표기 TOP 30


R132H     349
V600E     305
E545K     135
H1047R    128
R273C      82
E542K      68
R175H      66
R273H      60
R132C      59
R248Q      49
R213*      45
R248W      45
L42L       39
R282W      34
R196*      32
Y220C      32
K16fs      31
R80*       31
L35L       28
L100L      27
A2A        25
Q61R       25
Q269Q      25
H179R      25
L45L       24
P423P      24
R306*      24
G245S      24
E110K      24
L159L      23
Name: count, dtype: int64


변이 표기 예시 30개
['D623D', 'F736C', 'P117R', 'R305S', 'N23K', 'L192V', 'G592G', 'K277K', 'A699T', 'L111L', 'I136M', 'A219T', 'E237*', 'A746D', 'D465D', 'N1445D', 'T218T', 'G764G', 'E660Q', 'E291A', 'I382M', 'V412A', 'P222R', 'R104H', 'P17R', 'T1115M', 'L1854fs', 'R36H', 'R69Q', 'A372T']


In [54]:
# D. 한 유전자 셀에 여러 변이 표기가 기록된 경우 확인

# WT와 결측치를 제외한 변이 기록 추출
mutation_values = X.stack()
mutation_values = mutation_values[mutation_values != "WT"]

# 공백 기준으로 나뉘는 표기 개수 계산
token_counts = mutation_values.astype(str).str.split().str.len()

print("전체 변이 기록 셀 수:", len(mutation_values))
print("표기가 1개인 셀 수:", (token_counts == 1).sum())
print("표기가 2개 이상인 셀 수:", (token_counts >= 2).sum())
print("한 셀의 최대 표기 개수:", token_counts.max())

print("\n표기 개수별 셀 수")
display(token_counts.value_counts().sort_index().head(64))

print("\n표기가 2개 이상인 셀 예시 20개")
display(mutation_values[token_counts >= 2].head(20))

전체 변이 기록 셀 수: 218893
표기가 1개인 셀 수: 196867
표기가 2개 이상인 셀 수: 22026
한 셀의 최대 표기 개수: 64

표기 개수별 셀 수


1     196867
2      16261
3       3023
4       1399
5        382
6        374
7         79
8        159
9         25
10        82
11        16
12        63
13         6
14        40
15         5
16        28
17         2
18        19
19         1
20        19
21         1
22        12
24         8
26         6
28         3
30         4
32         2
36         2
38         1
39         1
44         1
60         1
64         1
Name: count, dtype: int64


표기가 2개 이상인 셀 예시 20개


2  ITGB5        Q369* I368N
   LAMB3        S622S G827R
   MYH4        E412K R1800C
   RYR1       D3546V G3739G
   TNC          A368V E646K
5  ADAMTS1      C521Y R258R
   ADCY9        G273R V155A
   ALDH3A1       F336L I71T
   ANO1        I758S P700fs
   BCAN         R107H E468E
   DIAPH3        Q924* P44P
   HDAC9         L99L N825N
   IRF2         K64T N167fs
   JAK2       Y435fs N193fs
   MLF1           H22H G49W
   MMP2         G179G A408A
   PAOX         S269S T286T
   PIAS3       V274M L226fs
   REV3L      Y2576F F1486F
   RHOBTB3      S372S T480T
dtype: str

In [55]:
# D. 전체 변이 표기 분리 및 유형별 분류 결과 확인

# 공백 기준으로 개별 변이 표기 분리
tokens = mutation_values.astype(str).str.split().explode()

# 변이 유형 분류
is_nonsense = tokens.str.fullmatch(r"[A-Z]\d+\*")
is_frameshift = tokens.str.contains("fs", case=False, regex=False)

is_substitution = tokens.str.fullmatch(r"[A-Z]\d+[A-Z]")

is_synonymous = (
    is_substitution
    & (tokens.str[0] == tokens.str[-1])
)

is_missense = is_substitution & ~is_synonymous

# 기타/미분류
is_other = ~(
    is_nonsense
    | is_frameshift
    | is_synonymous
    | is_missense
)

# 분류 결과 확인
print("전체 개별 변이 표기 수:", len(tokens))
print("미스센스:", is_missense.sum())
print("동의 변이:", is_synonymous.sum())
print("넌센스:", is_nonsense.sum())
print("프레임시프트 후보:", is_frameshift.sum())
print("기타/미분류:", is_other.sum())

print("\n분류 중복 개수:",
      (is_frameshift & (is_missense | is_synonymous | is_nonsense)).sum())

print("\n기타/미분류 표기 TOP 30")
display(tokens[is_other].value_counts().head(30))

전체 개별 변이 표기 수: 255164
미스센스: 164740
동의 변이: 66883
넌센스: 13289
프레임시프트 후보: 9914
기타/미분류: 338

분류 중복 개수: 0

기타/미분류 표기 TOP 30


*261*             3
*1232*            3
*459*             3
468_469LG>F*      2
312_313QY>HH      2
82_83DA>EI        2
139_140MG>I*      2
292_293DP>ES      2
127_128KL>KL      2
929_930LL>LL      2
435_436EA>DS      2
*399*             2
*271*             2
711_712FL>FL      2
*220*             2
594_595ME>IK      2
*902*             2
*626*             2
*235*             2
1105_1106PL>PL    2
187_188PL>PL      2
1441_1442AL>AL    2
*837*             2
2069_2070WA>*T    2
*455*             2
*194*             2
*141*             2
*924*             2
*335*             2
264_265MG>I*      2
Name: count, dtype: int64

#### 대표적인 기타 미분류의 변이 표기 
| 표기 | 해석 |
| :--- | :--- |
| `*261*` | 종결 코돈이 유지된 것으로 보이는 표기 |
| `312_313Q>HH` | 두 아미노산이 함께 치환된 표기 |
| `127_128KL>KL` | 두 아미노산이 동일하게 유지된 표기 |
| `468_469LG>F*` | 복합 아미노산 변화 및 종결 코돈 포함 |

#### 추가 피처명
| 변수명 | 설명 |
| :--- | :--- |
| `n_missense` | 단일 아미노산 치환 표기 수 |
| `n_synonymous` | 단일 아미노산 동의 변이 표기 수 |
| `n_nonsense` | 단일 아미노산이 종결 코돈으로 바뀐 표기 수 |
| `n_frameshift` | `fs`가 포함된 프레임시프트 후보 표기 수 |
| `n_other` | 나머지 복합 변이 및 미분류 표기 수 |

In [84]:
# D. 환자별 변이 유형 개수 피처 생성 함수

import re
import pandas as pd

def mutation_type_counts(X):
    # 변이 유형별 개수를 저장할 데이터프레임
    feature_names = [
        "n_missense",
        "n_synonymous",
        "n_nonsense",
        "n_frameshift",
        "n_other",
    ]

    result = pd.DataFrame(
        0,
        index=X.index,
        columns=feature_names,
        dtype="int32"
    )

    # WT와 결측치를 제외한 변이 기록 추출
    mutations = X.stack()
    mutations = mutations[mutations != "WT"]

    # 한 셀에 여러 변이가 있으면 공백 기준으로 분리
    tokens = mutations.astype(str).str.split().explode().dropna()

    # 개별 변이 표기를 유형별로 분류
    def classify(token):
        if "fs" in token.lower():
            return "n_frameshift"

        if re.fullmatch(r"[A-Z]\d+\*", token):
            return "n_nonsense"

        if re.fullmatch(r"[A-Z]\d+[A-Z]", token):
            if token[0] == token[-1]:
                return "n_synonymous"
            return "n_missense"

        return "n_other"

    types = tokens.map(classify)

    # 환자별 변이 유형 개수 집계
    counts = pd.crosstab(
        tokens.index.get_level_values(0),
        types
    )

    result = counts.reindex(
        index=X.index,
        columns=feature_names,
        fill_value=0
    ).astype("int32")

    return result

In [83]:
# D. 이진 인코딩 + 변이 개수 + 변이 유형별 개수 생성

type_counts = mutation_type_counts(X)

X_D = pd.concat([X_C, type_counts], axis=1)

print("C 방식 데이터 크기:", X_C.shape)
print("D 방식 데이터 크기:", X_D.shape)

display(type_counts.head(10))

C 방식 데이터 크기: (6201, 4385)
D 방식 데이터 크기: (6201, 4390)


col_0,n_missense,n_synonymous,n_nonsense,n_frameshift,n_other
0,17,7,1,0,0
1,9,5,0,3,0
2,79,39,6,0,0
3,3,3,1,0,0
4,23,11,0,0,0
5,164,81,11,31,0
6,2,1,0,1,0
7,0,1,0,0,0
8,7,2,0,0,0
9,12,6,2,0,0


In [58]:
# D. 변이 유형별 개수 피처의 합계 및 결측치 검증

# 환자별 변이 유형 개수 합계
total_type_counts = type_counts.sum(axis=1)

print("전체 변이 표기 수:", len(tokens))
print("유형별 개수 총합:", total_type_counts.sum())

print("전체 합계 일치 여부:",
      total_type_counts.sum() == len(tokens))

print("결측치 개수:", type_counts.isna().sum().sum())

print("변이 유전자 수보다 변이 표기 수가 적은 환자:",
      (total_type_counts < X_C["n_mutated_genes"]).sum())

print("\n변이 유형별 전체 개수")
display(type_counts.sum())

전체 변이 표기 수: 255164
유형별 개수 총합: 255164
전체 합계 일치 여부: True
결측치 개수: 0
변이 유전자 수보다 변이 표기 수가 적은 환자: 0

변이 유형별 전체 개수


col_0
n_missense      164740
n_synonymous     66883
n_nonsense       13289
n_frameshift      9914
n_other            338
dtype: int64

#### D + Multi-Hot

In [59]:
# D 피처와 Multi-Hot 피처를 결합하는 변환기

from scipy.sparse import hstack, csr_matrix
from sklearn.base import BaseEstimator, TransformerMixin

class DMultiHotTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, min_frequency=5):
        self.min_frequency = min_frequency

    def fit(self, X, y=None):
        # Multi-Hot 피처는 학습 fold에서만 선택
        self.multi_hot_ = MutationMultiHotEncoder(
            min_frequency=self.min_frequency
        )
        self.multi_hot_.fit(X)
        return self

    def transform(self, X):
        # 기존 D 피처 생성
        X_D = make_features_D(X)

        # Multi-Hot 피처 생성
        X_multi = self.multi_hot_.transform(X)

        # 두 피처를 희소행렬로 결합
        return hstack(
            [csr_matrix(X_D), X_multi],
            format="csr"
        )

## 3. 피처 구성별 모델 성능 비교

### 3-1. Pipeline 피처 변환기 구성 및 검증

In [60]:
# B·C·D 피처 구성을 위한 Pipeline 변환 함수

from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline

def make_features_B(X):
    return binarize_mutations(X)

def make_features_C(X):
    X_binary = binarize_mutations(X)

    return pd.concat([
        X_binary,
        mutation_count(X)
    ], axis=1)

def make_features_D(X):
    X_binary = binarize_mutations(X)

    return pd.concat([
        X_binary,
        mutation_count(X),
        mutation_type_counts(X)
    ], axis=1)

# Pipeline에서 사용할 변환기
transformer_B = FunctionTransformer(make_features_B)
transformer_C = FunctionTransformer(make_features_C)
transformer_D = FunctionTransformer(make_features_D)

# 기존 결과와 동일하게 생성되는지 확인
print("B 일치:", transformer_B.transform(X).equals(X_B))
print("C 일치:", transformer_C.transform(X).equals(X_C))
print("D 일치:", transformer_D.transform(X).equals(X_D))

B 일치: True
C 일치: True
D 일치: True


### 3-2. 공통 모델 설정

In [61]:
# XGBoost 정상 작동 확인

import xgboost as xgb

print("XGBoost 버전:", xgb.__version__)

XGBoost 버전: 3.4.1


In [62]:
# 팀 공통 Baseline 설정으로 B·C·D Pipeline 생성

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

base_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

pipeline_B = Pipeline([
    ("features", transformer_B),
    ("model", base_model)
])

pipeline_C = Pipeline([
    ("features", transformer_C),
    ("model", base_model)
])

pipeline_D = Pipeline([
    ("features", transformer_D),
    ("model", base_model)
])

print("B:", pipeline_B.named_steps["model"].get_params()["n_estimators"])
print("C:", pipeline_C.named_steps["model"].get_params()["n_estimators"])
print("D:", pipeline_D.named_steps["model"].get_params()["n_estimators"])

B: 200
C: 200
D: 200


### 3-3. A·B·C·D 교차검증 성능 비교

In [63]:
y_encoded, label_encoder = encode_target(y)

In [64]:
# 타깃 인코딩 결과 확인

print("타깃 데이터 크기:", y_encoded.shape)
print("클래스 개수:", len(label_encoder.classes_))
print("인코딩 결과:", y_encoded[:10])

타깃 데이터 크기: (6201,)
클래스 개수: 26
인코딩 결과: [ 8 19 20  9  6 21  2 23 12 21]


#### A 교차검증 성능비교

**min_frequency=5**

In [65]:
# A 방식: 유전자-변이 조합별 Multi-Hot Encoding
# 학습 fold에서 5회 이상 등장한 변이 조합만 사용
# B·C·D와 동일한 모델 및 교차검증 조건

cv_A = run_cv(
    pipeline_A,
    X,
    y_encoded,
    folds=folds
)

print(f"A 방식 CV Macro F1: {cv_A['mean']:.4f} ± {cv_A['std']:.4f}")

fold 0: Macro F1 = 0.1607
fold 1: Macro F1 = 0.1563
fold 2: Macro F1 = 0.1563
fold 3: Macro F1 = 0.1571
fold 4: Macro F1 = 0.1646
CV Macro F1 = 0.1590 (std 0.0032)
A 방식 CV Macro F1: 0.1590 ± 0.0032


**min_frequency=3**

In [66]:
# A 방식: Multi-Hot Encoding (최소 등장 빈도 3회)
# 기존 A(5회)와 동일한 XGBoost 및 5-Fold 사용

pipeline_A3 = Pipeline([
    ("features", MutationMultiHotEncoder(min_frequency=3)),
    ("model", clone(base_model))
])

cv_A3 = run_cv(
    pipeline_A3,
    X,
    y_encoded,
    folds=folds
)

print("\n[A 방식 - 빈도 3회]")
print(f"CV Macro F1: {cv_A3['mean']:.4f} ± {cv_A3['std']:.4f}")
print(f"Fold별 점수: {cv_A3['scores']}")

fold 0: Macro F1 = 0.1642
fold 1: Macro F1 = 0.1630
fold 2: Macro F1 = 0.1590
fold 3: Macro F1 = 0.1569
fold 4: Macro F1 = 0.1719
CV Macro F1 = 0.1630 (std 0.0052)

[A 방식 - 빈도 3회]
CV Macro F1: 0.1630 ± 0.0052
Fold별 점수: [0.1642363544334225, 0.16298444627743927, 0.15895921126101234, 0.15686993240078403, 0.17186307497096417]


#### B 교차검증 성능비교

In [67]:
# B. 이진 인코딩 피처의 5-Fold 교차검증

from src.validation import get_folds, run_cv

folds = get_folds(y_encoded)

result_B = run_cv(
    model=pipeline_B,
    X=X,
    y=y_encoded,
    folds=folds
)

print(f"\nB 평균 Macro F1: {result_B['mean']:.4f}")
print(f"B 표준편차: {result_B['std']:.4f}")

fold 0: Macro F1 = 0.3160
fold 1: Macro F1 = 0.3194
fold 2: Macro F1 = 0.3412
fold 3: Macro F1 = 0.3089
fold 4: Macro F1 = 0.3146
CV Macro F1 = 0.3200 (std 0.0111)

B 평균 Macro F1: 0.3200
B 표준편차: 0.0111


#### C 교차검증 성능비교

In [68]:
# C. 이진 인코딩 + 변이 유전자 수 피처의 5-Fold 교차검증

result_C = run_cv(
    model=pipeline_C,
    X=X,
    y=y_encoded,
    folds=folds
)

print(f"\nC 평균 Macro F1: {result_C['mean']:.4f}")
print(f"C 표준편차: {result_C['std']:.4f}")

# B 대비 성능 변화
diff_C = result_C["mean"] - result_B["mean"]
print(f"B 대비 Macro F1 변화: {diff_C:+.4f}")

fold 0: Macro F1 = 0.3562
fold 1: Macro F1 = 0.3794
fold 2: Macro F1 = 0.3715
fold 3: Macro F1 = 0.3550
fold 4: Macro F1 = 0.3692
CV Macro F1 = 0.3663 (std 0.0093)

C 평균 Macro F1: 0.3663
C 표준편차: 0.0093
B 대비 Macro F1 변화: +0.0462


#### D 교차검증 성능비교

**D**

In [69]:
# D. 이진 인코딩 + 변이 유전자 수 + 변이 유형별 개수의 5-Fold 교차검증

result_D = run_cv(
    model=pipeline_D,
    X=X,
    y=y_encoded,
    folds=folds
)

print(f"\nD 평균 Macro F1: {result_D['mean']:.4f}")
print(f"D 표준편차: {result_D['std']:.4f}")

# B 및 C 대비 성능 변화
diff_D_B = result_D["mean"] - result_B["mean"]
diff_D_C = result_D["mean"] - result_C["mean"]

print(f"B 대비 Macro F1 변화: {diff_D_B:+.4f}")
print(f"C 대비 Macro F1 변화: {diff_D_C:+.4f}")

fold 0: Macro F1 = 0.3814
fold 1: Macro F1 = 0.3891
fold 2: Macro F1 = 0.3821
fold 3: Macro F1 = 0.3820
fold 4: Macro F1 = 0.3832
CV Macro F1 = 0.3836 (std 0.0028)

D 평균 Macro F1: 0.3836
D 표준편차: 0.0028
B 대비 Macro F1 변화: +0.0635
C 대비 Macro F1 변화: +0.0173


**D + Multi-Hot (≥5회)**

In [70]:
# D + Multi-Hot (빈도 5회) 피처 결합 검증

# 첫 번째 fold의 학습·검증 데이터 분리
train_idx, valid_idx = folds[0]

X_train = X.iloc[train_idx]
X_valid = X.iloc[valid_idx]

# 학습 데이터에서만 Multi-Hot 피처 선택
transformer_DA5 = DMultiHotTransformer(min_frequency=5)
transformer_DA5.fit(X_train)

# 학습·검증 데이터 변환
X_train_DA5 = transformer_DA5.transform(X_train)
X_valid_DA5 = transformer_DA5.transform(X_valid)

print("기존 D 피처 수:", make_features_D(X_train).shape[1])
print("Multi-Hot 피처 수:", len(transformer_DA5.multi_hot_.feature_names_))
print("결합된 학습 데이터 크기:", X_train_DA5.shape)
print("결합된 검증 데이터 크기:", X_valid_DA5.shape)
print("데이터 형식:", type(X_train_DA5).__name__)

기존 D 피처 수: 4390
Multi-Hot 피처 수: 284
결합된 학습 데이터 크기: (4960, 4674)
결합된 검증 데이터 크기: (1241, 4674)
데이터 형식: csr_matrix


In [71]:
# D + Multi-Hot (빈도 5회) 5-Fold 교차검증
# 기존 B·C·D·A와 동일한 XGBoost 설정 사용

pipeline_DA5 = Pipeline([
    ("features", DMultiHotTransformer(min_frequency=5)),
    ("model", clone(base_model))
])

cv_DA5 = run_cv(
    pipeline_DA5,
    X,
    y_encoded,
    folds=folds
)

print("\n[D + Multi-Hot - 빈도 5회]")
print(f"CV Macro F1: {cv_DA5['mean']:.4f} ± {cv_DA5['std']:.4f}")
print(f"Fold별 점수: {cv_DA5['scores']}")

fold 0: Macro F1 = 0.3912
fold 1: Macro F1 = 0.4072
fold 2: Macro F1 = 0.3579
fold 3: Macro F1 = 0.3942
fold 4: Macro F1 = 0.3828
CV Macro F1 = 0.3867 (std 0.0164)

[D + Multi-Hot - 빈도 5회]
CV Macro F1: 0.3867 ± 0.0164
Fold별 점수: [0.39117251365620753, 0.4072360377363133, 0.3578558173128859, 0.3941882293145416, 0.3828249433841178]


**D + Multi-Hot (≥3회)**

In [35]:
# D + Multi-Hot (빈도 3회) 5-Fold 교차검증
# 기존 실험과 동일한 XGBoost 및 Fold 사용

pipeline_DA3 = Pipeline([
    ("features", DMultiHotTransformer(min_frequency=3)),
    ("model", clone(base_model))
])

cv_DA3 = run_cv(
    pipeline_DA3,
    X,
    y_encoded,
    folds=folds
)

print("\n[D + Multi-Hot - 빈도 3회]")
print(f"CV Macro F1: {cv_DA3['mean']:.4f} ± {cv_DA3['std']:.4f}")
print(f"Fold별 점수: {cv_DA3['scores']}")

fold 0: Macro F1 = 0.3939
fold 1: Macro F1 = 0.4051
fold 2: Macro F1 = 0.3589
fold 3: Macro F1 = 0.3831
fold 4: Macro F1 = 0.3847
CV Macro F1 = 0.3851 (std 0.0153)

[D + Multi-Hot - 빈도 3회]
CV Macro F1: 0.3851 ± 0.0153
Fold별 점수: [0.39385757033902524, 0.4050884889209338, 0.3589299845178407, 0.3831160453529247, 0.38472554502827083]


### 3-4. 실험 결과 기록 및 최종 피처 선정

In [72]:
# A·B·C·D 및 A, D + Multi-Hot 전체 실험 성능 비교표

import pandas as pd

comparison = pd.DataFrame({
    "피처 구성": [
        "A: Multi-Hot (≥5회)",
        "A: Multi-Hot (≥3회)",
        "B: 이진 인코딩",
        "C: 이진 + 변이 유전자 수",
        "D: 이진 + 변이 수 + 유형별 개수",
        "D + Multi-Hot (≥5회)",
        "D + Multi-Hot (≥3회)"
    ],

    "피처 수": [
        "Fold별 상이",
        "Fold별 상이",
        X_B.shape[1],
        X_C.shape[1],
        X_D.shape[1],
        "Fold별 상이",
        "Fold별 상이"
    ],

    "Macro F1 평균": [
        cv_A["mean"],
        cv_A3["mean"],
        result_B["mean"],
        result_C["mean"],
        result_D["mean"],
        cv_DA5["mean"],
        cv_DA3["mean"]
    ],

    "Macro F1 표준편차": [
        cv_A["std"],
        cv_A3["std"],
        result_B["std"],
        result_C["std"],
        result_D["std"],
        cv_DA5["std"],
        cv_DA3["std"]
    ]
})

# 기존 B 방식 대비 성능 개선 폭
comparison["B 대비 개선"] = (
    comparison["Macro F1 평균"] - result_B["mean"]
)

# 현재 최고 성능 순으로 정렬
comparison = comparison.sort_values(
    by="Macro F1 평균",
    ascending=False
).reset_index(drop=True)

display(comparison.round(4))

,피처 구성,피처 수,Macro F1 평균,Macro F1 표준편차,B 대비 개선
0,D + Multi-Hot (≥5회),Fold별 상이,0.3867,0.0164,0.0666
1,D + Multi-Hot (≥3회),Fold별 상이,0.3851,0.0153,0.0651
2,D: 이진 + 변이 수 + 유형별 개수,4390,0.3836,0.0028,0.0635
3,C: 이진 + 변이 유전자 수,4385,0.3663,0.0093,0.0462
4,B: 이진 인코딩,4384,0.3200,0.0111,0.0000
5,A: Multi-Hot (≥3회),Fold별 상이,0.1630,0.0052,-0.1571
6,A: Multi-Hot (≥5회),Fold별 상이,0.1590,0.0032,-0.1610


In [73]:
# 전체 피처 실험 결과를 팀 공통 형식으로 CSV에 기록
# 이미 기록된 실험은 중복 저장하지 않음

import pandas as pd

from src.evaluation import log_result
from src.config import EXPERIMENT_DIR

log_path = EXPERIMENT_DIR / "log_H_data.csv"

# 기존 실험 기록 확인
if log_path.exists():
    existing_log = pd.read_csv(log_path)
    recorded_names = set(existing_log["experiment"].dropna())
else:
    recorded_names = set()

model_params = pipeline_B.named_steps["model"].get_params()

experiments = [
    (
        "A_multihot_freq5",
        "유전자-변이 조합 Multi-Hot (빈도 ≥5회)",
        cv_A,
        "학습 Fold에서 5회 이상 등장한 조합만 사용"
    ),
    (
        "A_multihot_freq3",
        "유전자-변이 조합 Multi-Hot (빈도 ≥3회)",
        cv_A3,
        "학습 Fold에서 3회 이상 등장한 조합만 사용"
    ),
    (
        "B_binary",
        "4384개 유전자 이진 인코딩",
        result_B,
        "WT·결측=0, 변이=1"
    ),
    (
        "C_binary_mutation_count",
        "이진 인코딩 + n_mutated_genes",
        result_C,
        "환자별 변이 유전자 수 추가"
    ),
    (
        "D_binary_mutation_types",
        "이진 인코딩 + n_mutated_genes + 변이 유형별 개수 5개",
        result_D,
        "missense, synonymous, nonsense, frameshift, other 개수 추가"
    ),
    (
        "D_multihot_freq5",
        "D 피처 + Multi-Hot (빈도 ≥5회)",
        cv_DA5,
        "D 피처에 학습 Fold 기준 5회 이상 변이 조합 추가"
    ),
    (
        "D_multihot_freq3",
        "D 피처 + Multi-Hot (빈도 ≥3회)",
        cv_DA3,
        "D 피처에 학습 Fold 기준 3회 이상 변이 조합 추가"
    )
]

for experiment, features, cv, notes in experiments:

    if experiment in recorded_names:
        print(f"[건너뜀] {experiment}: 이미 기록됨")
        continue

    params = model_params.copy()

    if "freq5" in experiment:
        params["min_frequency"] = 5
    elif "freq3" in experiment:
        params["min_frequency"] = 3

    log_result(
        owner="H_data",
        notebook="02_features.ipynb",
        experiment=experiment,
        features=features,
        model="XGBoost",
        params=params,
        cv=cv,
        notes=notes
    )

    recorded_names.add(experiment)
    print(f"[기록 완료] {experiment}")

기록: log_H_data.csv ← A_multihot_freq5 (CV 0.1590)
[기록 완료] A_multihot_freq5
기록: log_H_data.csv ← A_multihot_freq3 (CV 0.1630)
[기록 완료] A_multihot_freq3
[건너뜀] B_binary: 이미 기록됨
[건너뜀] C_binary_mutation_count: 이미 기록됨
[건너뜀] D_binary_mutation_types: 이미 기록됨
기록: log_H_data.csv ← D_multihot_freq5 (CV 0.3867)
[기록 완료] D_multihot_freq5
기록: log_H_data.csv ← D_multihot_freq3 (CV 0.3851)
[기록 완료] D_multihot_freq3


In [74]:
# 실험 로그 저장 결과 확인

from src.config import EXPERIMENT_DIR

log_df = pd.read_csv(EXPERIMENT_DIR / "log_H_data.csv")

display(
    log_df[
        ["experiment", "features", "model", "cv_mean", "cv_std"]
    ]
)

,experiment,features,model,cv_mean,cv_std
0,B_binary,4384개 유전자 이진 인코딩,XGBoost,0.3200,0.0111
1,C_binary_mutation_count,이진 인코딩 + n_mutated_genes,XGBoost,0.3663,0.0093
2,D_binary_mutation_types,이진 인코딩 + n_mutated_genes + 변이 유형별 개수 5개,XGBoost,0.3836,0.0028
3,A_multihot_freq5,유전자-변이 조합 Multi-Hot (빈도 ≥5회),XGBoost,0.1590,0.0032
4,A_multihot_freq3,유전자-변이 조합 Multi-Hot (빈도 ≥3회),XGBoost,0.1630,0.0052
5,D_multihot_freq5,D 피처 + Multi-Hot (빈도 ≥5회),XGBoost,0.3867,0.0164
6,D_multihot_freq3,D 피처 + Multi-Hot (빈도 ≥3회),XGBoost,0.3851,0.0153


**최종 피처 선정 및 실험 결과 해석**

총 7개 피처 구성을 동일한 XGBoost 모델과 Stratified 5-Fold 교차검증으로 비교하였다.

- **최고 평균 성능:** D + Multi-Hot (빈도 ≥5회), Macro F1 0.3867
- **가장 안정적인 상위 후보:** D 단독, Macro F1 0.3836 ± 0.0028
- **Multi-Hot 단독:** Macro F1 0.1590~0.1630으로 이진 인코딩 기반 구성보다 낮은 성능을 보였다.
- **D + Multi-Hot 결합:** D 단독 대비 소폭 개선되었으나 Fold별 성능 변동이 증가하였다.

**최종 판단:** D + Multi-Hot (≥5회)를 최고 평균 성능 후보로 선정하고, D 단독을 안정적인 비교 후보로 유지한다. 두 모델의 OOF 예측 확률을 활용해 앙상블 효과를 추가 검증할 수 있다.

In [21]:
# src/features.py 수정 후 정상 작동 확인

import importlib
import src.features

# 수정된 함수 다시 불러오기
importlib.reload(src.features)

# 공통 함수로 변이 유형별 개수 생성
type_counts_shared = src.features.mutation_type_counts(X)

print("피처 크기:", type_counts_shared.shape)
print("결측치 개수:", type_counts_shared.isna().sum().sum())

print("\n변이 유형별 합계:")
print(type_counts_shared.sum())

피처 크기: (6201, 5)
결측치 개수: 0

변이 유형별 합계:
col_0
n_missense      164740
n_synonymous     66883
n_nonsense       13289
n_frameshift      9914
n_other            338
dtype: int64


### 3-5. OOF 및 test 예측 확률 저장

In [75]:
# 예측 저장을 위한 데이터 및 함수 확인
from src.evaluation import save_predictions
from src.data_loader import load_data
import inspect

# 팀 공통 예측 저장 함수의 인자 확인
print("save_predictions 인자:")
print(inspect.signature(save_predictions))

# 팀 공통 데이터 로더 확인
print("\nload_data 인자:")
print(inspect.signature(load_data))

# 현재 사용 중인 데이터 확인
print("\n학습 데이터 크기:", X.shape)
print("타깃 크기:", y_encoded.shape)
print("클래스 수:", len(label_encoder.classes_))

save_predictions 인자:
(owner: str, name: str, oof_proba, test_proba, classes, cv: dict, notes: str = '') -> None

load_data 인자:
(data_dir=PosixPath('/Users/admin/Documents/ah07-cancer-subtype/data'))

학습 데이터 크기: (6201, 4384)
타깃 크기: (6201,)
클래스 수: 26


In [77]:
# 팀 공통 데이터 로더로 test 데이터 불러오기

train, test = load_data()

# 데이터 내용은 확인하지 않고 구조만 확인
print("test 크기:", test.shape)
print("test 컬럼 수:", len(test.columns))

# 학습 데이터와 test의 피처 구성 및 순서 확인
test_features = test.drop(columns=["ID"])

print("학습·test 컬럼 일치:", X.columns.equals(test_features.columns))
print("test 행 수:", len(test_features))

test 크기: (2546, 4385)
test 컬럼 수: 4385
학습·test 컬럼 일치: True
test 행 수: 2546


In [80]:
# test 데이터의 변이 토큰 생성 과정 확인

mutations_test = test_features.stack()
mutations_test = mutations_test[mutations_test != "WT"]

tokens_test = mutations_test.astype(str).str.split().explode()

print("전체 토큰 수:", len(tokens_test))
print("결측 토큰 수:", tokens_test.isna().sum())

print("\n토큰 타입별 개수:")
print(tokens_test.map(type).value_counts())

전체 토큰 수: 337749
결측 토큰 수: 237

토큰 타입별 개수:
<class 'str'>      337512
<class 'float'>       237
Name: count, dtype: int64


In [85]:
# 수정된 전처리 함수가 test 데이터에서 정상 작동하는지 확인

test_type_counts = mutation_type_counts(test_features)

print("피처 크기:", test_type_counts.shape)
print("결측치 개수:", test_type_counts.isna().sum().sum())

피처 크기: (2546, 5)
결측치 개수: 0


In [86]:
# 기존 학습된 D 모델로 test 예측 재시도

test_proba_D = final_D.predict_proba(test_features)

print("OOF 확률 크기:", result_D["oof_proba"].shape)
print("test 확률 크기:", test_proba_D.shape)

OOF 확률 크기: (6201, 26)
test 확률 크기: (2546, 26)


In [87]:
# 수정 전후 D 학습 피처가 동일한지 확인

X_D_updated = make_features_D(X)

print("피처 크기 일치:", X_D.shape == X_D_updated.shape)
print("컬럼 구성 일치:", X_D.columns.equals(X_D_updated.columns))
print("전체 피처 값 일치:", X_D.equals(X_D_updated))

피처 크기 일치: True
컬럼 구성 일치: True
전체 피처 값 일치: True


In [91]:
# 전체 7개 실험의 OOF 및 test 예측 확률 저장

from pathlib import Path
from sklearn.base import clone
from src.evaluation import save_predictions
from src.config import EXPERIMENT_DIR

experiments_to_save = [
    ("A_multihot_freq5", pipeline_A, cv_A),
    ("A_multihot_freq3", pipeline_A3, cv_A3),
    ("B_binary", pipeline_B, result_B),
    ("C_binary_mutation_count", pipeline_C, result_C),
    ("D_binary_mutation_types", pipeline_D, result_D),
    ("D_multihot_freq5", pipeline_DA5, cv_DA5),
    ("D_multihot_freq3", pipeline_DA3, cv_DA3),
]

pred_dir = EXPERIMENT_DIR / "preds"

for name, pipeline, cv in experiments_to_save:

    # 저장된 예측 파일이 있는 경우 재학습 생략
    existing_files = list(pred_dir.glob(f"H_data__{name}*"))

    if existing_files:
        print(f"[건너뜀] {name}: 예측 파일 존재")
        continue

    print(f"\n===== {name} =====")

    # 전체 학습 데이터로 최종 모델 학습
    final_model = clone(pipeline)
    final_model.fit(X, y_encoded)

    # test 예측 확률 생성
    test_proba = final_model.predict_proba(test_features)

    # 확률 배열 크기 확인
    assert cv["oof_proba"].shape == (
        len(X), len(label_encoder.classes_)
    )
    assert test_proba.shape == (
        len(test_features), len(label_encoder.classes_)
    )

    # 팀 공통 형식으로 예측 확률 저장
    save_predictions(
        owner="H_data",
        name=name,
        oof_proba=cv["oof_proba"],
        test_proba=test_proba,
        classes=label_encoder.classes_,
        cv=cv,
        notes="Stratified 5-Fold, Macro F1"
    )

    print(f"[저장 완료] {name}")

print("\n전체 실험 예측 저장 확인 완료")

[건너뜀] A_multihot_freq5: 예측 파일 존재
[건너뜀] A_multihot_freq3: 예측 파일 존재
[건너뜀] B_binary: 예측 파일 존재
[건너뜀] C_binary_mutation_count: 예측 파일 존재
[건너뜀] D_binary_mutation_types: 예측 파일 존재
[건너뜀] D_multihot_freq5: 예측 파일 존재
[건너뜀] D_multihot_freq3: 예측 파일 존재

전체 실험 예측 저장 확인 완료


## 4. 번외 실험 추가

### 4-1. D + 아미노산 치환 쌍 피처 실험

#### 아미노산 치환 쌍 추출 함수 만들기

In [92]:
import re
from collections import Counter

def amino_acid_substitution_counts(X):
    """환자별 아미노산 치환 쌍의 발생 횟수를 계산"""

    pattern = re.compile(r"^([A-Z])\d+([A-Z])$")

    all_counts = []

    for _, row in X.iterrows():
        counts = Counter()

        for value in row:
            if pd.isna(value) or value == "WT":
                continue

            # 한 셀에 여러 변이가 있는 경우 각각 분리
            for mutation in str(value).split():
                match = pattern.fullmatch(mutation)

                if match:
                    before, after = match.groups()

                    # 동일 아미노산 치환은 제외
                    if before != after:
                        pair = f"{before}>{after}"
                        counts[pair] += 1

        all_counts.append(counts)

    return all_counts

In [93]:
sample_counts = amino_acid_substitution_counts(X.head(3))

for i, counts in enumerate(sample_counts):
    print(f"환자 {i + 1}: {dict(counts.most_common(10))}")

환자 1: {'P>R': 3, 'A>T': 2, 'I>M': 2, 'F>C': 1, 'R>S': 1, 'N>K': 1, 'L>V': 1, 'A>D': 1, 'N>D': 1, 'E>Q': 1}
환자 2: {'R>H': 2, 'T>M': 1, 'R>Q': 1, 'A>T': 1, 'A>V': 1, 'P>Q': 1, 'M>I': 1, 'K>M': 1}
환자 3: {'E>K': 15, 'P>S': 11, 'D>N': 8, 'S>F': 4, 'A>V': 4, 'R>C': 3, 'G>R': 3, 'G>E': 3, 'R>W': 2, 'R>Q': 2}


#### D + 치환 쌍 Transformer 구현

In [94]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import csr_matrix, hstack


class DAminoAcidTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, min_frequency=16):
        self.min_frequency = min_frequency

    def fit(self, X, y=None):
        # 학습 Fold의 환자별 치환 쌍 추출
        sample_counts = amino_acid_substitution_counts(X)

        # 환자별 중복을 제외한 출현 인원 계산
        patient_frequency = Counter(
            pair
            for counts in sample_counts
            for pair in counts.keys()
        )

        # 학습 Fold에서 기준 인원 이상 출현한 치환 쌍만 선택
        self.selected_pairs_ = sorted(
            pair
            for pair, freq in patient_frequency.items()
            if freq >= self.min_frequency
        )

        self.pair_to_idx_ = {
            pair: idx
            for idx, pair in enumerate(self.selected_pairs_)
        }

        return self

    def transform(self, X):
        # 기존 D 피처 생성
        X_D = make_features_D(X)

        # 환자별 치환 쌍 개수
        sample_counts = amino_acid_substitution_counts(X)

        rows, cols, values = [], [], []

        for row_idx, counts in enumerate(sample_counts):
            for pair, count in counts.items():
                col_idx = self.pair_to_idx_.get(pair)

                if col_idx is not None:
                    rows.append(row_idx)
                    cols.append(col_idx)
                    values.append(np.log1p(count))

        X_pairs = csr_matrix(
            (values, (rows, cols)),
            shape=(len(X), len(self.selected_pairs_)),
            dtype=np.float32
        )

        # D 피처 + 치환 쌍 피처 결합
        return hstack(
            [csr_matrix(X_D), X_pairs],
            format="csr"
        )

In [95]:
train_idx, val_idx = next(iter(folds))

X_train_fold = X.iloc[train_idx]
X_val_fold = X.iloc[val_idx]

aa_transformer = DAminoAcidTransformer(min_frequency=16)

X_train_aa = aa_transformer.fit_transform(X_train_fold)
X_val_aa = aa_transformer.transform(X_val_fold)

print("선택된 치환 쌍 수:", len(aa_transformer.selected_pairs_))
print("학습 데이터:", X_train_aa.shape)
print("검증 데이터:", X_val_aa.shape)
print("결측값 수:", np.isnan(X_train_aa.data).sum())
print("치환 쌍 예시:", aa_transformer.selected_pairs_[:10])

선택된 치환 쌍 수: 158
학습 데이터: (4960, 4548)
검증 데이터: (1241, 4548)
결측값 수: 0
치환 쌍 예시: ['A>D', 'A>E', 'A>G', 'A>P', 'A>S', 'A>T', 'A>V', 'C>F', 'C>G', 'C>R']


#### XGBoost 5-Fold 교차검증

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.base import clone

# D + 아미노산 치환 쌍 피처를 적용한 XGBoost Pipeline
pipeline_DAA = Pipeline([
    ("features", DAminoAcidTransformer(min_frequency=16)),
    ("model", clone(base_model))
])

# 기존 팀 공통 Fold와 평가 함수 사용
cv_DAA = run_cv(
    pipeline_DAA,
    X,
    y_encoded,
    folds=folds
)

# 기존 D 단독 모델과 비교
print("\n=== XGBoost 성능 비교 ===")
print(f"D 단독         : {result_D['mean']:.4f} ± {result_D['std']:.4f}")
print(f"D + 치환 쌍    : {cv_DAA['mean']:.4f} ± {cv_DAA['std']:.4f}")
print(f"성능 차이       : {cv_DAA['mean'] - result_D['mean']:+.4f}")

fold 0: Macro F1 = 0.4178
fold 1: Macro F1 = 0.4600
fold 2: Macro F1 = 0.4025
fold 3: Macro F1 = 0.4333
fold 4: Macro F1 = 0.4218
CV Macro F1 = 0.4271 (std 0.0192)

=== XGBoost 성능 비교 ===
D 단독         : 0.3836 ± 0.0028
D + 치환 쌍    : 0.4271 ± 0.0192
성능 차이       : +0.0435


#### 실험 로그 추가

In [97]:
from src.evaluation import log_result

log_result(
    owner="H_data",
    notebook="02_features.ipynb",
    experiment="D_amino_acid_pairs_freq16",
    features="D + amino acid substitution pairs (min_frequency=16, log1p)",
    model="XGBoost",
    params=base_model.get_params(),
    cv=cv_DAA,
    notes="Fold 내부 출현 인원 기준 피처 선택, D 대비 Macro F1 +0.0435"
)

print("8번째 실험 로그 저장 완료")

기록: log_H_data.csv ← D_amino_acid_pairs_freq16 (CV 0.4271)
8번째 실험 로그 저장 완료


#### 실험 예측 파일 저장

In [98]:
from sklearn.base import clone
from src.evaluation import save_predictions
from src.config import EXPERIMENT_DIR

name = "D_amino_acid_pairs_freq16"
pred_dir = EXPERIMENT_DIR / "preds"

# 이미 저장된 파일이 있는지 확인
expected_files = [
    pred_dir / f"H_data__{name}__oof.npy",
    pred_dir / f"H_data__{name}__test.npy",
    pred_dir / f"H_data__{name}__meta.json",
]

if all(path.exists() for path in expected_files):
    print(f"[건너뜀] {name}: 예측 파일 3개 존재")

else:
    # 전체 train으로 최종 모델 학습
    final_model = clone(pipeline_DAA)
    final_model.fit(X, y_encoded)

    # test 예측 확률 생성
    test_proba = final_model.predict_proba(test_features)

    # 데이터 크기 검증
    assert cv_DAA["oof_proba"].shape == (
        len(X), len(label_encoder.classes_)
    )
    assert test_proba.shape == (
        len(test_features), len(label_encoder.classes_)
    )

    # OOF 및 test 예측 저장
    save_predictions(
        owner="H_data",
        name=name,
        oof_proba=cv_DAA["oof_proba"],
        test_proba=test_proba,
        classes=label_encoder.classes_,
        cv=cv_DAA,
        notes="D + amino acid substitution pairs, min_frequency=16, log1p"
    )

    print(f"[저장 완료] {name}")

예측 저장: H_data__D_amino_acid_pairs_freq16 (CV 0.4271) → experiments/preds/
[저장 완료] D_amino_acid_pairs_freq16


#### 번외 포함 비교표

In [99]:
# 전체 8개 실험의 최종 성능 비교
final_results = [
    ("A Multi-Hot (5명 이상)", cv_A),
    ("A Multi-Hot (3명 이상)", cv_A3),
    ("B 이진 인코딩", result_B),
    ("C 변이 수 추가", result_C),
    ("D 변이 종류 추가", result_D),
    ("D + Multi-Hot (5명 이상)", cv_DA5),
    ("D + Multi-Hot (3명 이상)", cv_DA3),
    ("D + 아미노산 치환 쌍 (16명 이상)", cv_DAA),
]

final_comparison = pd.DataFrame([
    {
        "피처 구성": name,
        "Macro F1 평균": cv["mean"],
        "Macro F1 표준편차": cv["std"],
    }
    for name, cv in final_results
])

display(
    final_comparison
    .sort_values("Macro F1 평균", ascending=False)
    .reset_index(drop=True)
    .round(4)
)

,피처 구성,Macro F1 평균,Macro F1 표준편차
0,D + 아미노산 치환 쌍 (16명 이상),0.4271,0.0192
1,D + Multi-Hot (5명 이상),0.3867,0.0164
2,D + Multi-Hot (3명 이상),0.3851,0.0153
3,D 변이 종류 추가,0.3836,0.0028
4,C 변이 수 추가,0.3663,0.0093
5,B 이진 인코딩,0.3200,0.0111
6,A Multi-Hot (3명 이상),0.1630,0.0052
7,A Multi-Hot (5명 이상),0.1590,0.0032


**최종 피처 선정**

- 최종 선정: D + 아미노산 치환 쌍 (min_frequency=16)
- CV Macro F1: 0.4271 ± 0.0192
- 기존 D 대비 +0.0435 개선
- 치환 쌍은 학습 Fold 내부에서 선택하여 데이터 누수를 방지

### 4-2. E: 아미노산 치환 쌍 + 유전자별 끊김형 피처 검증 (LogisticRegression)

In [106]:
import numpy as np
import pandas as pd

from src.features import (
    mutation_count,
    mutation_type_counts,
    amino_acid_pair_counts,
    gene_truncation_flags,
)
from src.preprocessing import binarize_mutations


def make_features_E(X):
    X_binary = binarize_mutations(X)

    X_count = np.log1p(mutation_count(X))
    X_types = np.log1p(mutation_type_counts(X))
    X_pairs = np.log1p(amino_acid_pair_counts(X))
    X_trunc = gene_truncation_flags(X)

    return pd.concat(
        [X_binary, X_count, X_types, X_pairs, X_trunc],
        axis=1
    )


X_E_sample = make_features_E(X.head(3))

print("피처 크기:", X_E_sample.shape)
print("결측값 수:", X_E_sample.isna().sum().sum())
print("중복 컬럼 수:", X_E_sample.columns.duplicated().sum())

피처 크기: (3, 9174)
결측값 수: 0
중복 컬럼 수: 0


In [ ]:
import importlib
import src.features

importlib.reload(src.features)

print("새 함수 존재:", hasattr(src.features, "amino_acid_pair_counts"))
print("끊김형 함수 존재:", hasattr(src.features, "gene_truncation_flags"))

새 함수 존재: True
끊김형 함수 존재: True


In [ ]:
import importlib
import src.validation

importlib.reload(src.validation)

from src.validation import get_folds, run_cv

folds = get_folds(y_encoded)

print("Fold 수:", len(folds))
print("첫 번째 Fold 학습 샘플 수:", len(folds[0][0]))
print("첫 번째 Fold 검증 샘플 수:", len(folds[0][1]))
print("CV 함수 위치:", src.validation.__file__)

Fold 수: 5
첫 번째 Fold 학습 샘플 수: 4961
첫 번째 Fold 검증 샘플 수: 1240
CV 함수 위치: /Users/admin/Documents/ah07-cancer-subtype/src/validation.py


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression
from scipy.sparse import csr_matrix
from src.validation import get_folds, run_cv

# 피처 생성 후 희소행렬로 변환
def make_features_E_sparse(X):
    return csr_matrix(make_features_E(X).to_numpy(dtype=np.float32))

pipeline_E_LR = Pipeline([
    ("features", FunctionTransformer(make_features_E_sparse)),
    ("model", LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=3000,
        solver="lbfgs"
    ))
])

# 최신 팀 공통 Fold
folds = get_folds(y_encoded)

cv_E_LR = run_cv(
    pipeline_E_LR,
    X,
    y_encoded,
    folds=folds
)

print("=== 새 피처 E + LogisticRegression ===")
print(f"CV Macro F1: {cv_E_LR['mean']:.4f}")
print(f"표준편차: {cv_E_LR['std']:.4f}")
print("Fold별 점수:", cv_E_LR["scores"])

fold 1: Macro F1 = 0.4934
fold 2: Macro F1 = 0.4797
fold 3: Macro F1 = 0.4758
fold 4: Macro F1 = 0.5161
fold 5: Macro F1 = 0.4947
CV Macro F1 = 0.4919 (std 0.0141)
=== 새 피처 E + LogisticRegression ===
CV Macro F1: 0.4919
표준편차: 0.0141
Fold별 점수: [0.49335575210045995, 0.4797296792353784, 0.4758460427842044, 0.5160511136338943, 0.49469242822842524]
